# SBND RWM True Bucket: Results

- True bucket b per segment: shape and RMS with total errors (table), RMS over the segment time span and density b (figure), Run 1 and Run 2
- Reads `OUT_DIR/true_bucket.csv`, `true_bucket_density.npz` and the calibration CSVs (segment time spans)

## 1. Setup

- Set the paths in the Setup cell: `CALIB_CSV`, `OUT_DIR` (same as `errors.ipynb`)

In [ ]:
import os
import time
os.environ['TZ'] = 'UTC'
time.tzset()

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FormatStrFormatter, NullLocator, MultipleLocator
matplotlib.rcdefaults()

import sys
sys.path.insert(0, '../lib')
import paths
from segments import load_calibration
from ad_response import SEGMENTS
from true_bucket import load_npz
from plot_helpers import SEG_COLORS, FULL, STYLE, style_time_axis

# ===== Set these paths =====
CALIB_CSV = {1: '/path/to/run1_calibration.csv', 2: '/path/to/run2_calibration.csv'}   # calibration CSVs from SBND_RWM_CALIBRATION
OUT_DIR = '../output'   # output folder, the same in every notebook
paths.set_paths(calib=CALIB_CSV, out=OUT_DIR)
F_DENS = paths.out_file('true_bucket_density.npz')
F_CSV = paths.out_file('true_bucket.csv')
FIG_DIR = paths.out_file('figures')
os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update(STYLE)
RUNS = {'Run 1': dict(run=1, seg_labels={'before_a': 'Before Rotation A', 'before_c': 'Before Rotation C', 'during': 'Rotation', 'after': 'After Rotation'}),
        'Run 2': dict(run=2, seg_labels={'wide': 'Wide', 'normal_a': 'Normal A', 'normal_b': 'Normal B', 'narrow': 'Narrow'})}


## 2. True Bucket per Segment

- σL, σR, βL, βR of b and its RMS, each with its total error

In [ ]:
CSV = pd.read_csv(F_CSV, dtype={'ad_config': str}).set_index(['run', 'seg'])
DENS = load_npz(F_DENS)
pd.DataFrame({f"Run {r} {RUNS[f'Run {r}']['seg_labels'][s]}": {'AD config': CSV.loc[(r, s), 'ad_config'],
              **{p: f"{CSV.loc[(r, s), p]:.2f} ± {CSV.loc[(r, s), p + '_tot']:.2f}" for p in ('sl', 'sr', 'bl', 'br', 'rms')}}
              for r, s in SEGMENTS}).T

## 3. Figure

- Top: RMS of b per segment over its time span, band = total error; bottom: density b, mode at 0

In [ ]:
fig = plt.figure(figsize=(FULL, 5.0))
gs = fig.add_gridspec(2, 2, height_ratios=(1, 1.25), hspace=0.42, wspace=0.25)
for j, (name, r) in enumerate(RUNS.items()):
    d = load_calibration(r['run'])
    ax_t = fig.add_subplot(gs[0, j])
    ax_b = fig.add_subplot(gs[1, j])
    for seg, label in r['seg_labels'].items():
        g = d[d['seg'] == seg]
        a, b = g['start'].min(), g['end'].max()
        row = CSV.loc[(r['run'], seg)]
        c = SEG_COLORS[seg]
        ax_t.fill_between([a, b], row['rms'] - row['rms_tot'], row['rms'] + row['rms_tot'], color=c, alpha=0.25, lw=0)
        ax_t.plot([a, b], [row['rms']] * 2, color=c, lw=1.2)
        z = DENS[(r['run'], seg)]
        tt, ff = z['tt'], z['ff'] / (z['tt'][1] - z['tt'][0])
        ax_b.plot(tt, ff, color=c, lw=1.0, label=f"{label}: RMS {row['rms']:.2f} ns")
    t0, t1 = d.loc[d['seg'].isin(r['seg_labels']), 'start'].min(), d.loc[d['seg'].isin(r['seg_labels']), 'end'].max()
    style_time_axis(ax_t, (t0, t1), xlabel=None, fmt='%b\n%Y')
    ax_t.xaxis.set_major_locator(mdates.MonthLocator())
    ax_t.xaxis.set_minor_locator(NullLocator())
    ax_t.set_ylim(0.5, 2.3)
    ax_t.yaxis.set_major_locator(MultipleLocator(0.25))
    ax_t.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))
    ax_t.set_ylabel('RMS of $b$ [ns]')
    ax_t.set_title(f'SBND {name}', fontweight='bold', pad=3)
    ax_b.set_xlim(-6, 6)
    ax_b.set_ylim(0, 0.85)
    ax_b.set_xlabel('Time from the mode of $b$ [ns]')
    ax_b.set_ylabel('Density [1/ns]')
    ax_b.legend(loc='upper right', frameon=False, handlelength=1.4)
fig.savefig(f'{FIG_DIR}/true_bucket_results.png', dpi=300, bbox_inches='tight')
plt.show()